In [78]:
from bs4 import BeautifulSoup,Tag
import requests

url = "https://sicstus.sics.se/sicstus/docs/latest4/html/sicstus.html/index.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')  # or 'lxml'

contents = soup.select_one('body > div > ul.no-bullet')
print(contents)

<ul class="no-bullet">
<li><a href="Intro.html#Intro" id="toc-Introduction">Introduction</a></li>
<li><a href="Acknowledgments.html#Acknowledgments" id="toc-Acknowledgments-1">Acknowledgments</a></li>
<li><a href="Notation.html#Notation" id="toc-Notational-Conventions">1 Notational Conventions</a>
<ul class="no-bullet">
<li><a href="Keyboard-Characters.html#Keyboard-Characters" id="toc-Keyboard-Characters-1">1.1 Keyboard Characters</a></li>
<li><a href="Mode-Spec.html#Mode-Spec" id="toc-Mode-Spec-1">1.2 Mode Spec</a></li>
<li><a href="Development-and-Runtime-Systems.html#Development-and-Runtime-Systems" id="toc-Development-and-Runtime-Systems-1">1.3 Development and Runtime Systems</a></li>
<li><a href="Function-Prototypes.html#Function-Prototypes" id="toc-Function-Prototypes-1">1.4 Function Prototypes</a></li>
<li><a href="ISO-Compliance.html#ISO-Compliance" id="toc-ISO-Compliance-1">1.5 ISO Compliance</a></li>
</ul></li>
<li><a href="Glossary.html#Glossary" id="toc-Glossary-1">2 Gloss

In [79]:
def get_section(contents:Tag,section) -> Tag | None:
    for t in contents.descendants:
        if t.name == 'ul':
            possible_result = get_section(t,section)
            if possible_result:
                return possible_result
        elif t.name == 'li':
            if t.text.strip().split(' ')[0] == section:
                return t
        else:
            continue
    return None

In [80]:
built_in_predicates = get_section(contents,"11.3")
built_in_predicates
number_builtins = len(list(built_in_predicates.children))
number_builtins

3

In [81]:
teste = get_section(contents,"11.3.60")
teste

<li><a href="mpg_002dref_002ddebug.html#mpg_002dref_002ddebug" id="toc-debug_002f0---development">11.3.60 <code>debug/0</code>  <!-- /@w --> <i>development</i></a></li>

In [82]:
for c in built_in_predicates.children:
    print(c)

<a href="mpg_002dbpr.html#mpg_002dbpr" id="toc-Built_002dIn-Predicates">11.3 Built-In Predicates</a>


<ul class="no-bullet">
<li><a href="mpg_002dref_002dabolish.html#mpg_002dref_002dabolish" id="toc-abolish_002f_005b1_002c2_005d---ISO">11.3.1 <code>abolish/[1,2]</code>  <!-- /@w --> <i>ISO</i></a></li>
<li><a href="mpg_002dref_002dabort.html#mpg_002dref_002dabort" id="toc-abort_002f0">11.3.2 <code>abort/0</code></a></li>
<li><a href="mpg_002dref_002dabsolute_005ffile_005fname.html#mpg_002dref_002dabsolute_005ffile_005fname" id="toc-absolute_005ffile_005fname_002f_005b2_002c3_005d---hookable">11.3.3 <code>absolute_file_name/[2,3]</code>  <!-- /@w --> <i>hookable</i></a></li>
<li><a href="mpg_002dref_002dacyclic_005fterm.html#mpg_002dref_002dacyclic_005fterm" id="toc-acyclic_005fterm_002f1---ISO">11.3.4 <code>acyclic_term/1</code>  <!-- /@w --> <i>ISO</i></a></li>
<li><a href="mpg_002dref_002dadd_005fbreakpoint.html#mpg_002dref_002dadd_005fbreakpoint" id="toc-add_005fbreakpoint_002f2--

In [83]:
member_page_link = "https://sicstus.sics.se/sicstus/docs/latest4/html/sicstus.html/mpg_002dref_002dmember.html"
response = requests.get(member_page_link)
soup = BeautifulSoup(response.text, 'html.parser')  # or 'lxml'
soup

<!DOCTYPE html PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">

<html>
<!-- Created by GNU Texinfo 6.7, http://www.gnu.org/software/texinfo/ -->
<head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<title>mpg-ref-member (SICStus Prolog)</title>
<meta content="mpg-ref-member (SICStus Prolog)" name="description"/>
<meta content="mpg-ref-member (SICStus Prolog)" name="keywords"/>
<meta content="document" name="resource-type"/>
<meta content="global" name="distribution"/>
<meta content="makeinfo" name="Generator"/>
<link href="index.html" rel="start" title="Top"/>
<link href="Predicate-Index.html" rel="index" title="Predicate Index"/>
<link href="index.html#SEC_Contents" rel="contents" title="Table of Contents"/>
<link href="mpg_002dbpr.html" rel="up" title="mpg-bpr"/>
<link href="mpg_002dref_002dmemberchk.html" rel="next" title="mpg-ref-memberchk"/>
<link href="mpg_002dref_002dload_005fforeign_005fresource.html" rel="prev" tit

In [107]:
def concat_ps(ps:list[Tag])->str:
    p_tags = [p for p in ps if p.name =='p']
    return "\n".join(p.text for p in p_tags)

class PredicateDocBuilder:
    def __init__(self):
        self.compat = None
        self.signature = None
        self.templates = []
        self.synopsis =None
        self.description = None
        self.arguments = None
        self.examples = None
        self.see_also = None
        self.exception = None
        self.backtracking =  None
        self.comments = None
        self.tips = None
        self.tags = []
        pass
    def handle_synopsis(self,contents):
        name, arity = self.signature
        # TODO if arity is list there will be more templates
        n_templates = 1
        contents = [c for c in contents if c.name is not None]
        if type(arity) is list:
            n_templates = len(arity)
        assert len(contents) > (n_templates), f"In predicate {name}/{arity} the synopsis is probably missing information"
        for i in range(n_templates):
            template_p : Tag= contents[i]
            self.templates.append(template_p.get_text(strip=True))
        self.synopsis = concat_ps(contents[n_templates:])

    def handle_description(self,contents):
        self.description =  concat_ps(contents)
    def handle_comments(self,contents):
        self.comments =  concat_ps(contents)

    def handle_tips(self,contents):
        self.tips =  concat_ps(contents)

    def handle_arguments(self,contents):
        for i in range(0,len(contents),2):
            term = contents[i]
            definition = contents[i+1]
        return 
    def handle_examples(self,contents):
        return 
    def handle_see_also(self,contents):
        self.see_also = concat_ps(contents)
    def handle_exception(self,contents):
        self.exceptions= concat_ps(contents)
    def handle_backtracking(self,contents):
        self.backtracking= concat_ps(contents)
    
    def handle_keywords(self,predicate_name):
        tags = predicate_name.select('i')
        for t in tags:
            self.tags.append(t.text)


    def handle_predicate_name(self,predicate_name):
        signature = predicate_name.text
        parts = signature.split('/')
        assert len(parts) == 2

        arity = parts[1]
        if parts[1][0] == '[':
            numbers = parts[1][1:-1].split(',')
            if all(n.isdigit() for n in numbers):
                arity = [int(n) for n in numbers]
            else:
                print(arity)
        elif parts[1].isdigit():
            arity = int(parts[1])
        self.signature = (parts[0],arity)
    
    def to_pldoc(self):
        r = ""
        for t in self.templates:
            r +=  "%!    " + t  + "\n"
        r += "%\n"
        for line in self.synopsis.split('\n'):
            line= line.strip()
            if len(line) > 0:
                r +=  "%     " + line  + "\n"
        return r

    def build(self,page):
        predicate_name = page.select("h4.subsection")
        assert len(predicate_name) == 1, "Found a page without predicate name"
        self.handle_predicate_name(predicate_name[0].select("code")[0])
        
        self.handle_keywords(predicate_name[0])

        subsections = page.select("h4.subheading")

        section_functions =  {
            'Synopsis': self.handle_synopsis,
            'Description': self.handle_description,
            'Arguments': self.handle_arguments,
            'Examples': self.handle_examples,
            'See Also': self.handle_see_also,
            'Exceptions': self.handle_exception,
            'Comments': self.handle_comments,
            'Tips': self.handle_tips,
            'Backtracking': self.handle_backtracking,
        }

        for section in subsections:
            name = section.text
            if name in section_functions:
                section_functions[name](self.section_contents(section))
            else:
                print("No registered function to handle:",name)
                raise TypeError

    def section_contents(self,tag:Tag)-> list[Tag]:
        result = []
        current_symbol = tag.next_sibling
        while current_symbol is not None and current_symbol.name != "h4":
            result.append(current_symbol)
            current_symbol = current_symbol.next_sibling
        return result
    def print(self):
        if self.compat != None:
            print(f"compat:{self.compat}")
        if self.signature != None:
            print(f"name:{self.signature}")
        if self.synopsis !=None:
            print(f"synopsis:{self.synopsis}")
        if self.description != None:
            print(f"description:{self.description}")
        if self.arguments != None:
            print(f"arguments:{self.arguments}")
        if self.examples != None:
            print(f"examples:{self.examples}")
        if self.see_also != None:
            print(f"see_also:{self.see_also}")
        if self.exception != None:
            print(f"exception:{self.exception}")
        if self.backtracking !=  None:
            print(f"backtracking:{self.backtracking}")
        if self.tags != []:
            print(f"tags:{self.tags}")
    


        

In [108]:
predicate_builder = PredicateDocBuilder()
predicate_builder.build(soup)
predicate_builder.print()


name:('member', 2)
synopsis:is true if Element occurs in the List.  It may be used
to test for an element or to enumerate all the elements by backtracking.
Indeed, it may be used to generate the List!

description:In the context of this predicate, a term occurs in a list if it can be
unified with an element of the list.

see_also:ref-lte-acl,
library(lists).

backtracking:On backtracking, an attempt is made to unify Element with successive
elements of List.
If List is not a proper list, then on backtracking it is unified with
lists of ever increasing length.



In [109]:
import os
def get_predicate_page(predicate_li: Tag):
    name = predicate_li.code.text.split('/')[0]
    
    print(f"Predicate {name}, found")
    file_path = f'builtins/{name}.html'
    if os.path.exists(file_path):
        result = ""
        with open(file_path,'r') as f:
            result = f.read()
        return result
    
    print(f"Predicate {name}, not stored")
    member_page_link = predicate_li.select('a')[0].get('href')
    base_link = "https://sicstus.sics.se/sicstus/docs/latest4/html/sicstus.html/"
    response = requests.get(base_link + member_page_link)
    if response.status_code != 200:
        raise Exception(response.status_code)
    with open(file_path) as f:
        f.write(response.text)
    return response.text

def visit_predicate(predicate_li:Tag):

    page = get_predicate_page(predicate_li)
    soup = BeautifulSoup(page, 'html.parser')  # or 'lxml'
    predicate_builder = PredicateDocBuilder()
    predicate_builder.build(soup)
    return predicate_builder

builtin_predicates = built_in_predicates.select("ul")[0].children

result = []
for predicate in builtin_predicates:
    if predicate.name is None:
        continue
    elif predicate.name == 'li':
        p = visit_predicate(predicate)
        result.append(p)
    elif predicate.name == 'ul':
        print(predicate)
    else:
        print(predicate)


Predicate abolish, found
Predicate abort, found
Predicate absolute_file_name, found
Predicate acyclic_term, found
Predicate add_breakpoint, found
Predicate ,, found
Predicate append, found
Predicate arg, found
Predicate ask_query, found
Predicate assert, found
Predicate asserta, found
Predicate assertz, found
Predicate at_end_of_line, found
Predicate at_end_of_stream, found
Predicate atom, found
Predicate atom_chars, found
Predicate atom_codes, found
Predicate atom_concat, found
Predicate atom_length, found
Predicate atomic, found
Predicate bagof, found
Predicate bb_delete, found
Predicate bb_get, found
Predicate bb_put, found
Predicate bb_update, found
Predicate block, found
Predicate break, found
Predicate breakpoint_expansion, found
Predicate byte_count, found
Predicate call, found
[1,2,...,255]
Predicate call_cleanup, found
Predicate call_residue_vars, found
Predicate callable, found
Predicate catch, found
Predicate char_code, found
Predicate char_conversion, found
Predicate charac

In [110]:
names = set()
for r in result:
    n = r.signature[0]
    if n in names:
        print(n)
    names.add(r.signature[0])
print(len(names))
print(len(result))

256
256


In [113]:
with open('builtins.pl','w') as f:
    for predicate in result:
        f.write(predicate.to_pldoc())
        f.write('\n')

In [ ]:
print(result[0].to_pldoc())

%!    abolish(+Predicates)
%!    abolish(+Predicates,+Options)
%
%     Removes procedures from the Prolog database.

